not the production system, this is my investigation lab


# Apartment Pricing Scraper (Phase 1)

## Objective
Extract floor plan rents and specials from apartment websites and convert them into structured data for analysis.

## Current Focus
- Single property scraping (Nora Seattle)
- Identify pricing source (HTML vs API vs JS-rendered)
- Build first structured dataset

In [34]:
# imports
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [35]:
from urllib.parse import urljoin

## Step 1: Define target website

In [36]:
# Nora URL
url = "https://liveatnora.com/floor-plans/"

## Step 2: Fetch raw HTML (first attempt)

In [37]:
response = requests.get(url)
print(response.status_code)

200


## Step 3: Save raw HTML for inspection (data engineering best practice)

In [38]:
html = response.text

with open("../data/raw/nora_floorplans.html", "w", encoding="utf-8") as f:
    f.write(html)

## Step 4: Parse HTML with BeautifulSoup

In [39]:
soup = BeautifulSoup(html, "lxml")

## Step 5: Basic validation checks

In [40]:
print(soup.title.text)

Student Housing Seattle | Floor Plans | Nora


## Step 6: Attempt to locate pricing in raw HTML
(If nothing appears, data is likely loaded dynamically via API)

In [41]:
text = soup.get_text()

found = False

for line in text.split("\n"):
    if "$" in line:
        print(line.strip())
        found = True

if not found:
    print("No pricing found in static HTML → likely API or JS-rendered content")

No pricing found in static HTML → likely API or JS-rendered content


## Step 7: Look for embedded JSON or hidden structured data

In [43]:
scripts = soup.find_all("script")

for i, s in enumerate(scripts[:10]):
    if s.string:
        if "rent" in s.string.lower() or "floor" in s.string.lower():
            print(f"\n--- Script {i} ---\n")
            print(s.string[:1000])


--- Script 4 ---

(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
})(window,document,'script','dataLayer','GTM-WBSHLJG');

--- Script 6 ---


{
  "@context": "https://schema.org",
  "@graph": [
    {
      "@type": "ApartmentComplex",
      "@id": "https://liveatnora.com/",
      "additionalType": "LocalBusiness",
      "name": "NORA Apartments",
      "telephone": "206-704-5368",
      "address": {
        "@type": "PostalAddress",
        "streetAddress": "4106 12th Ave. NE",
        "addressLocality": "Seattle",
        "addressRegion": "WA",
        "postalCode": "98105"
      },
      "numberOfBedrooms": {
        "@type": "QuantitativeValue",
        "minValue": 0,
        "maxValue": 2
      },
      "accommodationFloorPlan": [
        {
      

## Step 8: Next step (pending DevTools inspection)
- If API exists → switch to requests-based JSON extraction
- If JS-rendered → move to Playwright
- If embedded → parse script JSON

In [42]:
# saving the raw HTML
with open("../data/raw/nora_floorplans.html", "w", encoding="utf-8") as f:
    f.write(html)